<a href="https://colab.research.google.com/github/AktanM11/AI-OI/blob/main/DAY2_WEEK2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install qdrant-client

In [ ]:
pip install fastembed

In [27]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Document
from qdrant_client.http import models
from qdrant_client.http.exceptions import UnexpectedResponse
from fastembed import TextEmbedding

In [32]:
COLLECTION_NAME = "my_knowledge_base"
MODEL_NAME = "BAAI/bge-small-en-v1.5"

def get_model_dimension(model_name: str) -> int:
    print(f"модель {model_name}")
    embedding_model = TextEmbedding(model_name=model_name)
    info = next(embedding_model.embed(["test"]))
    dimension = len(info)
    print(f"Размерность модели: {dimension}")
    return dimension

def init_qdrant_collection():
    client = QdrantClient(url=ENDPOINT, api_key=API_KEY)

    vector_size = get_model_dimension(MODEL_NAME)
    distance_metric = models.Distance.COSINE

    try:
        existing_collection = client.get_collection(collection_name=COLLECTION_NAME)
        current_config = existing_collection.config.params.vectors

        if isinstance(current_config, models.VectorParams):
            existing_size = current_config.size
            existing_distance = current_config.distance
        else:
            existing_size = current_config[""].size
            existing_distance = current_config[""].distance

        if existing_size == vector_size and existing_distance == distance_metric:
            print(f"Коллекция '{COLLECTION_NAME}' уже существует (Размерность: {existing_size}). Пересоздание не требуется")
            return
        else:
            print(f"Параметры не совпадают. В базе: size={existing_size}. Ожидается: size={vector_size}. Пересоздаем")
            client.delete_collection(collection_name=COLLECTION_NAME)

    except UnexpectedResponse as e:
        if e.status_code == 404:
            print(f"Коллекция '{COLLECTION_NAME}' не найдена.")
        else:
            raise e

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=vector_size,
            distance=distance_metric
        ),
        hnsw_config=models.HnswConfigDiff(
            m=16,
            ef_construct=100,
        )
    )
    print(f"Коллекция '{COLLECTION_NAME}' создана в облаке")

In [33]:
init_qdrant_collection()

модель BAAI/bge-small-en-v1.5
Размерность модели: 384
Коллекция 'my_knowledge_base' не найдена.
Коллекция 'my_knowledge_base' создана в облаке


In [34]:
init_qdrant_collection()

модель BAAI/bge-small-en-v1.5
Размерность модели: 384
Коллекция 'my_knowledge_base' уже существует (Размерность: 384). Пересоздание не требуется
